In [ ]:
# Import necessary libraries

import os

from langchain_community.utilities import SQLDatabase

from langchain_classic.chains import create_sql_query_chain

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_google_genai import ChatGoogleGenerativeAI

: 

In [30]:
# Connect your MySQL database
# Make sure to install the required packages
host = 'localhost'
port = '3306'
username = 'root'
password = '5998'
database_schema = 'text_to_sql'
mysql_uri = f"mysql+pymysql://{username}:{password}@{host}:{port}/{database_schema}"
db = SQLDatabase.from_uri(mysql_uri, sample_rows_in_table_info=2)

In [31]:
# Database connection
db = SQLDatabase.from_uri(mysql_uri, sample_rows_in_table_info=1)

In [32]:
# create a llm propt template
# Create the LLM Prompt Template
from langchain_core.prompts import ChatPromptTemplate

template = """Based on the table schema below, write a SQL query that would answer the user's question:
Remember : Only provide me the sql query dont include anything else. Provide me sql query in a single line dont add line breaks
Table Schema: {schema}
Question: {question}
SQL Query:
"""

prompt = ChatPromptTemplate.from_template(template)

In [33]:
# get the schema of the database
def get_schema(db):
    schema = db.get_table_info()
    return schema

In [34]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

In [35]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=api_key
)

In [36]:
# Create the SQL query chain using the LLM and the prompt template

sql_chain = (
    RunnablePassthrough.assign(schema=lambda _:get_schema(db)) 
    | prompt
    | llm.bind(stop=["\nSQLResult:"])
    | StrOutputParser()
)

In [37]:
# test the SQL query chain with a sample question

resp = sql_chain.invoke({"question": "What is the total 'Line Total' for Geiss Company"})
print(resp)

SELECT SUM(t1.`Line Total`) FROM sales_order AS t1 JOIN customers AS t2 ON t1.`Customer Name Index` = t2.`Customer Index` WHERE t2.`Customer Names` = 'Geiss Company';


In [38]:
#test the SQL query chain with a sample question
resp=sql_chain.invoke({"question": "What was the budget of Product 12"})
print(resp)

SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12'


In [39]:
import re

query = re.search(r"```sql\s*(.*?)\s*```", resp, re.DOTALL | re.IGNORECASE)

if query:
    query=query.group(1).strip()

In [40]:
import re

question = "Show all records from the sales_order table"

schema = db.get_table_info()

response = llm.invoke(
    prompt.format(
        schema=schema,
        question=question
    )
)

resp = response.content

# Extract text from Gemini response
if isinstance(resp, list):
    if len(resp) > 0 and isinstance(resp[0], dict):
        resp = resp[0].get("text", "")
    else:
        resp = str(resp)

# Convert to string
resp = str(resp)

# Extract SQL from markdown code block if present
match = re.search(
    r"```(?:sql)?\s*(.*?)\s*```",
    resp,
    re.DOTALL | re.IGNORECASE
)

if match:
    query = match.group(1).strip()
else:
    query = resp.strip()

print("Generated SQL Query:")
print(query)

Generated SQL Query:
SELECT * FROM sales_order


### RAGAS Implementation


In [48]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

In [ ]:
evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

In [47]:
pip show ragas langchain langchain-core langchain-community

Name: ragas
Version: 0.4.3
Summary: Evaluation framework for RAG and LLM applications
Home-page: https://github.com/vibrantlabsai/ragas
Author: 
Author-email: 
License: Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by